# S1 · Arquitectura Big Data

**Integrante** · Joel Jonatan Quispe Mamani

**Dimensión** · Velocidad mediana de las aeronaves en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

## Mi aporte

En U1 uso PySpark sobre el Parquet de OpenSky extraído con Trino. En U2 el mismo dominio se observa con OpenSky REST, Kafka, Spark Structured Streaming y Grafana. Lambda permite mantener el cálculo histórico y actualizar el indicador con capturas nuevas.

In [4]:
from pathlib import Path

from pyspark.sql import SparkSession

bronze = Path("/opt/s01-arquitectura/data/2026")
bloque = bronze / "01" / "2026-01-01_2026-01-05.parquet"

spark = (
    SparkSession.builder.master("local[2]")
    .appName("AeroStreamPeru-S01")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
print(f"Spark {spark.version}")

Spark 4.2.0


In [5]:
estados = spark.read.parquet(str(bloque))
estados.printSchema()
print(f"Filas: {estados.count():,} · Columnas: {len(estados.columns)}")
from pyspark.sql import functions as F

estados.select("time", "icao24", "onground", "velocity", "baroaltitude").show(5)
estados.filter(F.col("onground") == F.lit(False)).agg(
    F.count("*").alias("observaciones_en_vuelo"),
    F.count("velocity").alias("observaciones_con_velocidad"),
).show()

root
 |-- time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vertrate: double (nullable = true)
 |-- callsign: string (nullable = true)
 |-- onground: boolean (nullable = true)
 |-- alert: boolean (nullable = true)
 |-- spi: boolean (nullable = true)
 |-- squawk: string (nullable = true)
 |-- baroaltitude: double (nullable = true)
 |-- geoaltitude: double (nullable = true)
 |-- lastposupdate: double (nullable = true)
 |-- lastcontact: double (nullable = true)
 |-- hour: long (nullable = true)

Filas: 2,365,631 · Columnas: 17
+----------+------+--------+------------------+------------+
|      time|icao24|onground|          velocity|baroaltitude|
+----------+------+--------+------------------+------------+
|1767466277|e8047a|   false|229.59941756235077|     10668.0|
|1767466278|e8047a|   false|229.59941756235077|     106

## Interpretación

`velocity` registra la velocidad sobre el suelo en m/s. Las filas con velocidad faltante se ven en la diferencia entre los dos conteos. La disponibilidad de las mediciones ayuda a interpretar la mediana.

In [6]:
spark.stop()